In [1]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("AU Mic") 
#print(search_result)
lc2min = search_result[4].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

In [2]:

# ============================================================
# PASSO 2: CARREGAR MÁSCARA DE FLARES E TRÂNSITOS (TXT EDITÁVEL)
# ============================================================
print("\n" + "="*60)
print("ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# Arquivo editável com colunas: t_ini, t_fim
txt_path = "intervalos_flares_transitos_editavel.txt"

try:
    # Lê como CSV simples (separado por vírgula), mas em arquivo .txt
    df_intervalos = pd.read_csv(txt_path, sep=",", comment="#")

    # Validação básica das colunas esperadas
    colunas_esperadas = {"t_ini", "t_fim"}
    if not colunas_esperadas.issubset(df_intervalos.columns):
        raise ValueError(f"Arquivo deve conter as colunas {colunas_esperadas}. Colunas encontradas: {set(df_intervalos.columns)}")

    # Converte para lista de pares [inicio, fim]
    mascara_flares_list = df_intervalos[["t_ini", "t_fim"]].dropna().values.tolist()

    print(f"✓ {len(mascara_flares_list)} intervalo(s) carregado(s) de '{txt_path}':")
    for i, (ini, fim) in enumerate(mascara_flares_list, start=1):
        print(f"  {i}. [{ini:.6f}, {fim:.6f}]")

except Exception as e:
    print(f"✗ Erro ao carregar '{txt_path}': {e}")
    print("→ Usando lista vazia.")
    mascara_flares_list = []

# Criar máscara booleana
mascara_flares = np.zeros(len(t), dtype=bool)
for ini, fim in mascara_flares_list:
    mascara_flares |= (t >= ini) & (t <= fim)

mask_good_flares = ~mascara_flares

print(f"\nMáscara criada:")
print(f"  Pontos excluídos (flares/trânsitos): {mascara_flares.sum()}")
print(f"  Pontos para ajuste: {mask_good_flares.sum()}")



ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT
✓ 25 intervalo(s) carregado(s) de 'intervalos_flares_transitos_editavel.txt':
  1. [3883.239461, 3883.334868]
  2. [3884.751868, 3884.843937]
  3. [3885.024950, 3885.109542]
  4. [3885.549959, 3885.617899]
  5. [3886.130183, 3886.411659]
  6. [3888.081298, 3888.115210]
  7. [3888.784350, 3888.805991]
  8. [3889.179949, 3889.251286]
  9. [3889.826802, 3889.854535]
  10. [3891.013277, 3891.076262]
  11. [3891.741181, 3891.795356]
  12. [3891.844004, 3891.908792]
  13. [3891.950028, 3892.001562]
  14. [3892.631057, 3892.697367]
  15. [3892.812488, 3892.900441]
  16. [3893.017865, 3893.101673]
  17. [3893.403291, 3893.458281]
  18. [3896.687407, 3896.849926]
  19. [3897.567644, 3897.779646]
  20. [3899.153300, 3899.218797]
  21. [3899.542373, 3899.616046]
  22. [3900.664900, 3900.748400]
  23. [3904.512267, 3904.614076]
  24. [3904.782061, 3904.847219]
  25. [3905.340994, 3905.482584]

Máscara criada:
  Pontos excluídos (flares/t

In [15]:
from astropy.stats import sigma_clip
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from lightkurve import LightCurve

from scipy.interpolate import CubicSpline
from scipy.ndimage import gaussian_filter1d

# ============================================================
# PASSO 3: AJUSTE POLINOMIAL + FLATTEN LOCAL + SPLINE LOCAL
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Polinomial com Segmentos + Spline em Regiões")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# REGIÕES MANUAIS (poly / flatten)
# ============================================================

ajustes_manuais = [
    [3883.0050, 3883.4554, "poly", 4, 2.5],
    [3883.1409, 3883.7849, "poly", 4, 2.5],
    [3883.8264, 3884.0955, "poly", 4, 2.5],
    [3884.2828, 3885.3000, "poly", 4, 2.5],
    [3885.2839, 3885.4326, "poly", 4, 2.5],
    [3885.6205, 3885.7794, "poly", 2, 2.5],
    [3892.6280, 3892.7040, "poly", 1, 2.5],
    [3894.2089, 3896.5099, "poly", 4, 2.0],

    #[3899.86607, 3900.8691, "poly", 6, 2.5],
    #[3900.8273, 3900.8691, "flatten", None, None],
]

# ============================================================
# REGIÕES SPLINE  <-- ADICIONE / EDITE AQUI
# Formato: [t_inicio, t_fim, nbins, grau_k, fator_s, sigma_clip_interno]
# ============================================================

regioes_spline = [
    # [t_ini,      t_fim,   nbins, k, fator_s, sigma_clip_int]
    #[3899.86607, 3900.8691,  190,  4, 0.0001,  1.5],  # região com flares
    #[3900.8273,  3901.44,     60,  4, 0.001,   1.5],  # região mais tranquila
    # [3905.0,   3907.00,     60,  3, 0.01,    2.2],  # descomente para mais
    [3899.9880, 3900.996, 100, 4, 0.9, 1.2]    #[3894.1100, 3896.5099, 40, 2, 0.05, 1.0],  # AJUSTADO: menos bins, grau menor, mais suavização

    #[3899.501, 3900.826,   60,  7,  0.01,   1.5],  # mais suave, ignora flares

    #[3900.8273,  3901.44,     30,  3,  0.01,   1.5],
]


# ============================================================
# REGIÕES FLATTEN LOCAL  <-- ADICIONE / EDITE AQUI
# Formato: [t_inicio, t_fim, window_length, polyorder, sigma, break_tolerance, niters]
# Exemplo: [3900.8273, 3900.9512, 320, 3, 2.5, 10, 4]
# ============================================================

regioes_flatten = [
    #[3900.5909, 3900.9512, 280, 1, 1.5, 1, 4],
    # [3887.1500, 3888.4400, 280, 2, 2.2, 8, 5],
]

# ============================================================
# FLATTEN GLOBAL (fallback para uso em ajustes_manuais do tipo flatten)
# ============================================================

flcd, trend = lc2_m.flatten(
    window_length=320,
    polyorder=3,
    return_trend=True,
    break_tolerance=10,
    niters=4,
    sigma=2.5,
    mask=mask_good_flares
)

# ============================================================
# AJUSTE AUTOMÁTICO
# ============================================================

N_SEGMENTOS_AUTO = 40
GRAU_AUTO        = 4
SIGMA_AUTO       = 3.0

# ============================================================
# FUNÇÕES
# ============================================================

def fitting_segment(t_seg, f_seg, mask_seg, deg, sigma):
    if len(t_seg) < deg + 2:
        return np.full_like(f_seg, np.nanmedian(f_seg))
    t_mid = np.median(t_seg)
    t_s   = t_seg - t_mid
    good  = mask_seg.copy()
    modelo = np.full_like(f_seg, np.nan)
    for _ in range(5):
        if np.sum(good) < deg + 2:
            break
        coef = np.polyfit(t_s[good], f_seg[good], deg=deg)
        modelo_good = np.polyval(coef, t_s[good])
        modelo = np.interp(t_s, t_s[good], modelo_good)
        resid = f_seg - modelo
        clipped = sigma_clip(resid[good], sigma=sigma, maxiters=1)
        if clipped.mask is np.ma.nomask:
            break
        good[np.where(good)[0]] = ~clipped.mask
    if np.all(np.isnan(modelo)):
        modelo = np.full_like(f_seg, np.nanmedian(f_seg))
    return modelo


def fitting_spline(t_seg, f_seg, mask_seg, nbins, k, fator_s, sigma_clip_interno=1.8, clip_iters=5):
    t_ok = t_seg[mask_seg]
    f_ok = f_seg[mask_seg]

    if len(t_ok) < k + 2:
        print("    [spline] pontos insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    # clip robusto a outliers (mad_std nao eh inflado pela cauda do flare)
    if sigma_clip_interno is not None and len(f_ok) > k + 2:
        cl = sigma_clip(
            f_ok, sigma=sigma_clip_interno, maxiters=clip_iters,
            cenfunc='median', stdfunc='mad_std'
        )
        t_ok = t_ok[~cl.mask]
        f_ok = f_ok[~cl.mask]

    if len(t_ok) < k + 2:
        print("    [spline] pontos insuficientes apos sigma_clip, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    edges = np.linspace(t_ok.min(), t_ok.max(), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    bin_t, bin_f = [], []
    for i in range(nbins):
        in_bin = (t_ok >= edges[i]) & (t_ok < edges[i + 1])
        if np.sum(in_bin) >= 1:
            bin_t.append(centers[i])
            bin_f.append(np.nanmedian(f_ok[in_bin]))

    bin_t = np.array(bin_t)
    bin_f = np.array(bin_f)

    if len(bin_t) < k + 2:
        print("    [spline] bins insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    # segunda passada: remove BINS anomalos (ex: cauda de flare que escapou do filtro ponto a ponto)
    if len(bin_f) > k + 3:
        cl_bins = sigma_clip(bin_f, sigma=2.0, maxiters=3, cenfunc='median', stdfunc='mad_std')
        if np.sum(~cl_bins.mask) >= k + 2:
            n_removidos = np.sum(cl_bins.mask)
            if n_removidos > 0:
                print(f"    [spline] {n_removidos} bin(s) anomalo(s) removido(s) antes do ajuste.")
            bin_t = bin_t[~cl_bins.mask]
            bin_f = bin_f[~cl_bins.mask]

    s_val = fator_s * len(bin_t) * np.nanvar(bin_f) if fator_s > 0 else 0.0
    spline = UnivariateSpline(bin_t, bin_f, k=k, s=s_val, ext=3)
    modelo = spline(t_seg)

    med = np.nanmedian(f_ok)
    std_ok = np.nanstd(f_ok)
    limite = 6 * std_ok if std_ok > 0 else 6 * abs(med) + 1e-6
    fora = np.abs(modelo - med) > limite
    if np.any(fora):
        print(f"    [spline] {np.sum(fora)} ponto(s) fora do limite seguro, substituindo.")
        bons = ~fora
        if np.sum(bons) >= 2:
            modelo[fora] = np.interp(t_seg[fora], t_seg[bons], modelo[bons])
        else:
            modelo[fora] = med

    return modelo


def costurar_bordas(t, modelo, bordas, dados_observados=None, tamanho_janela=30, sigma_gauss=10):
    """
    dados_observados : array booleano (True onde há dados reais, False nos gaps)
                       Se None, aplica suavização em tudo (comportamento anterior)
    """
    modelo_costurado = np.copy(modelo).astype(float)
    print(f"\nAplicando costura em {len(bordas)} bordas...")

    for borda_idx in bordas:
        inicio = max(0, borda_idx - tamanho_janela)
        fim = min(len(t) - 1, borda_idx + tamanho_janela)

        if fim - inicio < 4:
            tempos = t[inicio:fim + 1]
            transicao = np.interp(tempos, [t[inicio], t[fim]],
                                  [modelo_costurado[inicio], modelo_costurado[fim]])
            modelo_costurado[inicio:fim + 1] = transicao
            continue

        n = len(t)
        ancora_esq = max(0, inicio - tamanho_janela // 2)
        ancora_dir = min(n - 1, fim + tamanho_janela // 2)

        idx_ancora = [ancora_esq, inicio, fim, ancora_dir]
        t_ancora   = t[idx_ancora]
        f_ancora   = modelo_costurado[idx_ancora]

        cs = CubicSpline(t_ancora, f_ancora, bc_type='natural')

        tempos_zona = t[inicio:fim + 1]
        centro = t[borda_idx]
        largura = (t[fim] - t[inicio]) / 2 + 1e-10
        peso = 0.5 * (1 - np.cos(np.pi * (1 - np.abs(tempos_zona - centro) / largura)))

        spline_vals = cs(tempos_zona)
        modelo_costurado[inicio:fim + 1] = (
            peso * spline_vals + (1 - peso) * modelo_costurado[inicio:fim + 1]
        )

    # Suavização gaussiana APENAS nas regiões com dados reais
    if sigma_gauss > 0:
        if dados_observados is not None:
            # Encontra segmentos contínuos com dados e suaviza cada um separadamente
            modelo_suavizado = np.copy(modelo_costurado)
            em_segmento = False
            seg_inicio = 0

            for i in range(len(t)):
                tem_dado = dados_observados[i]

                if tem_dado and not em_segmento:
                    seg_inicio = i
                    em_segmento = True
                elif (not tem_dado or i == len(t) - 1) and em_segmento:
                    seg_fim = i if not tem_dado else i + 1
                    segmento = modelo_costurado[seg_inicio:seg_fim]
                    if len(segmento) > 1:
                        modelo_suavizado[seg_inicio:seg_fim] = gaussian_filter1d(
                            segmento, sigma=sigma_gauss
                        )
                    em_segmento = False

            modelo_costurado = modelo_suavizado
        else:
            # Sem máscara: suaviza tudo (comportamento anterior)
            modelo_costurado = gaussian_filter1d(modelo_costurado, sigma=sigma_gauss)

    return modelo_costurado
# ============================================================
# AJUSTE AUTOMÁTICO BASE
# ============================================================

modelo_manchas = np.zeros_like(f)
bordas_indices = set()
edges_auto = np.linspace(t.min(), t.max(), N_SEGMENTOS_AUTO + 1)
segmentos = [(edges_auto[i], edges_auto[i + 1]) for i in range(N_SEGMENTOS_AUTO)]

print("Ajuste automático...")
for i, (ini, fim) in enumerate(segmentos):
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue
    modelo_manchas[idx] = fitting_segment(
        t[idx], f[idx], mask_good_flares[idx], GRAU_AUTO, SIGMA_AUTO
    )
    if i > 0:
        bordas_indices.add(np.where(idx)[0][0])

# ============================================================
# AJUSTES MANUAIS (poly / flatten)
# ============================================================

print(f"Aplicando {len(ajustes_manuais)} ajustes manuais...")
for ini, fim, metodo, grau, sig in ajustes_manuais:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue

    if metodo == "poly":
        modelo_manchas[idx] = fitting_segment(
            t[idx], f[idx], mask_good_flares[idx], grau, sig
        )

    elif metodo == "flatten":
        print(f"  FLATTEN (global) {ini:.4f} - {fim:.4f}")
        modelo_manchas[idx] = trend.flux.value[idx]

        idx_inicio = np.where(idx)[0][0]
        janela = 15
        i0 = max(0, idx_inicio - janela)
        i1 = min(len(modelo_manchas) - 1, idx_inicio + janela)
        modelo_manchas[i0:i1 + 1] = np.linspace(
            modelo_manchas[i0], modelo_manchas[i1], i1 - i0 + 1
        )

    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# ============================================================
# AJUSTES FLATTEN LOCAL POR REGIÃO (configuração completa por [t_i, t_f])
# ============================================================

print(f"\nAplicando {len(regioes_flatten)} região(ões) flatten local...")
for ini, fim, wlen, pord, sig, btol, niter in regioes_flatten:
    regiao = (t >= ini) & (t <= fim)
    if not np.any(regiao):
        print(f"  [flatten] nenhum ponto em {ini:.4f} - {fim:.4f}")
        continue

    # True = ignora no flatten. Usa somente pontos bons dentro da região.
    mask_flatten_regiao = (~regiao) | (~mask_good_flares)

    print(
        f"  [flatten] {ini:.4f} - {fim:.4f} | "
        f"window={wlen}, poly={pord}, sigma={sig}, "
        f"break_tol={btol}, niters={niter}"
    )

    flcd_r, trend_r = lc2_m.flatten(
        window_length=int(wlen),
        polyorder=int(pord),
        return_trend=True,
        break_tolerance=float(btol),
        niters=int(niter),
        sigma=float(sig),
        mask=mask_flatten_regiao
    )

    modelo_manchas[regiao] = trend_r.flux.value[regiao]
    bordas_indices.add(np.where(regiao)[0][0])
    bordas_indices.add(np.where(regiao)[0][-1])

# ============================================================
# AJUSTES SPLINE POR REGIÃO (aplicados por último)
# ============================================================

print(f"\nAplicando {len(regioes_spline)} região(ões) spline...")
for ini, fim, nbins, k, fator_s, sigma_clip_int in regioes_spline:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        print(f"  [spline] nenhum ponto em {ini:.4f} - {fim:.4f}")
        continue

    print(
        f"  [spline] {ini:.4f} - {fim:.4f} | "
        f"nbins={nbins}, k={k}, fator_s={fator_s}, sigma_clip={sigma_clip_int}"
    )

    modelo_manchas[idx] = fitting_spline(
        t[idx], f[idx], mask_good_flares[idx],
        nbins=nbins, k=k, fator_s=fator_s,
        sigma_clip_interno=sigma_clip_int
    )

    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# ============================================================
# COSTURA FINAL — junta auto + manual + flatten local + spline local
# ============================================================

# ============================================================
# LIMPEZA DE BORDAS INTERNAS (evita costura falsa dentro de regiões já cobertas)
# ============================================================

def limpar_bordas_internas(t, bordas_indices, regioes_overlay, margem=0.001):
    """
    Remove bordas que caem ESTRITAMENTE dentro de uma região de override
    (manual/flatten/spline), mantendo apenas bordas que estão fora dessas
    regiões ou exatamente nas extremidades delas.
    """
    bordas_validas = set()
    for idx in bordas_indices:
        tempo = t[idx]
        dentro_sem_ser_borda = False
        for ini, fim in regioes_overlay:
            if (tempo > ini + margem) and (tempo < fim - margem):
                dentro_sem_ser_borda = True
                break
        if not dentro_sem_ser_borda:
            bordas_validas.add(idx)
    return bordas_validas

regioes_overlay = (
    [(r[0], r[1]) for r in ajustes_manuais] +
    [(r[0], r[1]) for r in regioes_flatten] +
    [(r[0], r[1]) for r in regioes_spline]
)

n_antes = len(bordas_indices)
bordas_indices = limpar_bordas_internas(t, bordas_indices, regioes_overlay)
print(f"\nBordas internas removidas: {n_antes - len(bordas_indices)} (restaram {len(bordas_indices)})")

# ============================================================
# COSTURA FINAL — junta auto + manual + flatten local + spline local
# ============================================================

# DEPOIS — substitua pelo wrapper + chamada
def costurar_com_gaps(t, modelo, bordas, limite_gap_fator=5, tamanho_janela=30, sigma_gauss=10):
    dt_mediano = np.median(np.diff(t))
    limite_gap = dt_mediano * limite_gap_fator
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]

    modelo_costurado = costurar_bordas(
        t, modelo, bordas,
        dados_observados=None,
        tamanho_janela=tamanho_janela,
        sigma_gauss=0  # sem gauss aqui
    )

    if sigma_gauss > 0:
        for i0, i1 in zip(seg_inicios, seg_fins):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)

    return modelo_costurado

modelo_manchas_suave = costurar_com_gaps(
    t, modelo_manchas, sorted(bordas_indices),
    limite_gap_fator=5,
    tamanho_janela=30,
    sigma_gauss=10
)

# ============================================================
# RESÍDUO
# ============================================================

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

# ============================================================
# GRÁFICOS
# ============================================================

%matplotlib qt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(
    t,
    f,
    'k.-',
    ms=1.5,
    lw=0.5,
    alpha=0.6,
    label='Dados'
)
ax1.plot(
    t,
    modelo_manchas_suave,
    color='red',
    lw=2.2,
    label='Modelo Final'
)

for ini, fim, *_ in regioes_spline:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.10, label='_spline')
for ini, fim, *_ in regioes_flatten:
    ax1.axvspan(ini, fim, color='mediumseagreen', alpha=0.10, label='_flatten')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(
    t,
    residual_manchas,
    'b.-',
    ms=1.5,
    lw=0.5,
    alpha=0.7
)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_spline:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.10)
for ini, fim, *_ in regioes_flatten:
    ax2.axvspan(ini, fim, color='mediumseagreen', alpha=0.10)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Polinomial com Segmentos + Spline em Regiões
Ajuste automático...
Aplicando 8 ajustes manuais...

Aplicando 0 região(ões) flatten local...

Aplicando 1 região(ões) spline...
  [spline] 3899.9880 - 3900.9960 | nbins=100, k=4, fator_s=0.9, sigma_clip=1.2

Bordas internas removidas: 14 (restaram 40)

Aplicando costura em 40 bordas...

OK — Ajuste concluído.


In [20]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# VERIFICAÇÕES
# ============================================================

required = [
    "t",
    "f",
    "modelo_manchas_suave",
    "residual_manchas"
]

for var in required:
    if var not in globals():
        raise ValueError(f"Variável '{var}' não encontrada.")

# ============================================================
# PARÂMETROS AU MIC b
# ============================================================

T0_b  = globals().get("T0_b", 1330.39051)
P_b   = globals().get("P_b", 8.463000)
dur_b = globals().get("dur_b", 3.50/24)

# ============================================================
# PARÂMETROS AU MIC c
# ============================================================

T0_c  = globals().get("T0_c", 1342.2223)
P_c   = globals().get("P_c", 18.859019)
dur_c = globals().get("dur_c", 4.50/24)

# ============================================================
# REGIÕES DO PIPELINE
# ============================================================

regioes_flatten_plot = globals().get("regioes_flatten", [])
regioes_spline_plot  = globals().get("regioes_spline", [])
mascara_plot         = globals().get("mascara_flares_list", [])

# ============================================================
# FUNÇÃO DOS TRÂNSITOS
# ============================================================

def transit_times(T0, P, tmin, tmax):

    nmin = int(np.ceil((tmin - T0)/P))
    nmax = int(np.floor((tmax - T0)/P))

    return [
        T0 + n*P
        for n in range(nmin, nmax + 1)
    ]

# ============================================================
# TRÂNSITOS VISÍVEIS
# ============================================================

centers_b = transit_times(
    T0_b,
    P_b,
    np.nanmin(t),
    np.nanmax(t)
)

centers_c = transit_times(
    T0_c,
    P_c,
    np.nanmin(t),
    np.nanmax(t)
)

print("\n==============================")
print("TRÂNSITOS NO SETOR")
print("==============================")

print(f"\nAU Mic b ({len(centers_b)})")
for tc in centers_b:
    print(f"BTJD = {tc:.5f}")

print(f"\nAU Mic c ({len(centers_c)})")
for tc in centers_c:
    print(f"BTJD = {tc:.5f}")

# ============================================================
# FIGURA
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(18,10),
    sharex=True
)

# ============================================================
# PAINEL SUPERIOR
# ============================================================

ax1.plot(
    t,
    f,
    'k.-',
    ms=1.5,
    alpha=0.45,
    label='Dados'
)

ax1.plot(
    t,
    modelo_manchas_suave,
    color='red',
    lw=2.5,
    label='Modelo Final'
)

# ------------------------------------------------------------
# Máscara
# ------------------------------------------------------------

first = True

for ini, fim in mascara_plot:

    ax1.axvspan(
        ini,
        fim,
        color='orange',
        alpha=0.22,
        label='Máscara flare/trânsito'
        if first else "_nolegend_"
    )

    first = False

# ------------------------------------------------------------
# Flatten
# ------------------------------------------------------------

first = True

for reg in regioes_flatten_plot:

    ini, fim = reg[:2]

    ax1.axvspan(
        ini,
        fim,
        color='mediumseagreen',
        alpha=0.15,
        label='Flatten local'
        if first else "_nolegend_"
    )

    first = False

# ------------------------------------------------------------
# Spline
# ------------------------------------------------------------

first = True

for reg in regioes_spline_plot:

    ini, fim = reg[:2]

    ax1.axvspan(
        ini,
        fim,
        color='dodgerblue',
        alpha=0.10,
        label='Spline local'
        if first else "_nolegend_"
    )

    first = False

# ------------------------------------------------------------
# AU Mic b
# ------------------------------------------------------------

first = True

for tc in centers_b:

    ax1.axvspan(
        tc-dur_b/2,
        tc+dur_b/2,
        color='royalblue',
        alpha=0.22,
        label='AU Mic b'
        if first else "_nolegend_"
    )

    ax1.axvline(
        tc,
        color='royalblue',
        ls='--',
        lw=1.2
    )

    first = False

# ------------------------------------------------------------
# AU Mic c
# ------------------------------------------------------------

first = True

for tc in centers_c:

    ax1.axvspan(
        tc-dur_c/2,
        tc+dur_c/2,
        color='seagreen',
        alpha=0.22,
        label='AU Mic c'
        if first else "_nolegend_"
    )

    ax1.axvline(
        tc,
        color='seagreen',
        ls='--',
        lw=1.2
    )

    first = False

ax1.set_ylabel("Fluxo")
ax1.set_title(
    "AU Mic - Modelo Final + Regiões Especiais + Trânsitos",
    fontsize=15,
    fontweight='bold'
)

ax1.grid(alpha=0.25)
ax1.legend(
    fontsize=9,
    ncol=2,
    loc='upper right'
)

# ============================================================
# PAINEL INFERIOR
# ============================================================

ax2.plot(
    t,
    residual_manchas,
    'k-',
    lw=0.5,
    alpha=0.55,
    label='Residual'
)

ax2.axhline(
    1,
    color='black',
    ls='--'
)

for ini, fim in mascara_plot:
    ax2.axvspan(
        ini,
        fim,
        color='orange',
        alpha=0.22
    )

for reg in regioes_flatten_plot:

    ini, fim = reg[:2]

    ax2.axvspan(
        ini,
        fim,
        color='mediumseagreen',
        alpha=0.15
    )

for reg in regioes_spline_plot:

    ini, fim = reg[:2]

    ax2.axvspan(
        ini,
        fim,
        color='dodgerblue',
        alpha=0.10
    )

ax2.set_xlabel("Tempo [BTJD]")
ax2.set_ylabel("Residual")
ax2.grid(alpha=0.25)
ax2.legend()

plt.tight_layout()
plt.show()


TRÂNSITOS NO SETOR

AU Mic b (3)
BTJD = 3886.21651
BTJD = 3894.67951
BTJD = 3903.14251

AU Mic c (2)
BTJD = 3888.18986
BTJD = 3907.04888
